## Aim:

This notebook takes a directory of DEM tiles (imported from Copernicus Data Explorer, or DEFRA (Uk.gov)), and exports a merged DEM .tif.

AI helped, but did NOT generate, parts of the logic in this code. Specifically, it helped with the tempfile functionality, as this was new to me.

This code has been run every time I need to merge Digital Elevation Model Tiles into one raster.

In [1]:
import tempfile, os
import numpy as np
import rasterio
from rasterio.merge import merge
from pathlib import Path
import glob

dem_files = glob.glob('../../data/LizardDEM/*.tif')

OUT_PATH = Path('../../data/LizardDEM/out')
OUT_PATH.mkdir(parents=True, exist_ok=True)

NODATA = -9999
clipped_paths = []

with tempfile.TemporaryDirectory() as tmpdir:
    for f in dem_files:
        out_name = Path(f).stem + '_clean.tif'
        out_path = os.path.join(tmpdir, out_name)

        with rasterio.open(f) as src:
            data = src.read(1).astype('float32')

            if src.nodata is not None:
                data[data == src.nodata] = NODATA

            # also catch the -1e38 sentinel even if nodata isn't set
            data[data < -9000] = NODATA

            profile = src.profile.copy()
            profile.update(dtype='float32', nodata=NODATA)

            with rasterio.open(out_path, 'w', **profile) as dst:
                dst.write(data, 1)

        clipped_paths.append(out_path)

    # Merge
    datasets = [rasterio.open(p) for p in clipped_paths]
    mosaic, transform = merge(datasets, nodata=NODATA, method='max')

    profile = datasets[0].profile
    profile.update(
        width=mosaic.shape[2],
        height=mosaic.shape[1],
        transform=transform,
        dtype='float32',
        nodata=NODATA,
    )

    with rasterio.open(OUT_PATH / 'WellsDEM.tif', 'w', **profile) as dst:
        dst.write(mosaic)

    for ds in datasets:
        ds.close()

print("Saved")

Saved
